## Imports

In [53]:
# GIS and spatial data libraries
import rasterio
from rasterio.crs import CRS
from rasterio.plot import show
from rasterio.transform import xy
from rasterio.warp import reproject, Resampling, transform_bounds

import pyproj
from pyproj import Transformer

from affine import Affine
from osgeo import gdal
gdal.UseExceptions()
from shapely.geometry import box

# Data analysis and machine learning libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from itertools import combinations
from scipy.stats import skew
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Data access libraries
import ee
import requests

import json
import os
import pip
import platform
import time

from pathlib import Path
from IPython.display import Image, display

# Check for load errors
print(
    "All libraries imported successfully\n" 
    f"gdal v.{gdal.__version__}\n"
    f"pip v.{pip.__version__}\n"
    f"Python v{platform.python_version()}"
)

All libraries imported successfully
gdal v.3.13.3
pip v.25.2
Python v3.13.7


In [ ]:
# Verify paths
print(
    f"Working directory: {Path.cwd()}\n" 
    f"Python executable: {os.sys.executable}\n"
    f"Datasets folder exists: {Path("Datasets").exists()}\n"
    f"Assets folder exists: {Path("Assets").exists()}\n"
)

## EarthEngine Sentinel-2 & Dynamic World V1


### EE Authentication and AOI


In [11]:
# EE Authentication
ee.Authenticate()
ee.Initialize(project="gis-project-510419")

In [13]:
# Load AOI bounds saved by the DEM notebook
with (Path("Assets") / "dem_AOI.geojson").open(encoding="utf-8") as f:
    gj = json.load(f)

xs, ys = zip(*gj["features"][0]["geometry"]["coordinates"][0])
xmin_geo, ymin_geo, xmax_geo, ymax_geo = min(xs), min(ys), max(xs), max(ys)

print(f"AOI: {xmin_geo, ymin_geo, xmax_geo, ymax_geo}")

AOI: (-123.29161663788408, 49.25498431356825, -122.88478957342635, 49.55071922714218)


In [14]:
# EE AOI, _geo set in DEM bound transforms
aoi = ee.Geometry.Rectangle([
    xmin_geo,
    ymin_geo ,
    xmax_geo,
    ymax_geo
])

### Get Harmonized Sentinel-2 MSI Level-2A (SR)

In [ ]:
# Get image from Google Earth Engine
def get_image_collection(year, aoi_bounds, tile, max_cloudy_pixels):
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi_bounds)
        .filterDate(f"{year}-08-01",f"{year}-09-01")
        .filter(ee.Filter.eq('MGRS_TILE', tile))
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", max_cloudy_pixels))
    )

# Helper Functions for metadata
def add_band_names(image):
    return image.set("band_names", image.bandNames())

def get_crs(image):
    return image.set("crs", image.select("B4").projection().crs())

# SR Scale 
def scale_reflectance(image):
    scale_factor = 0.0001 # from GEE "The assets contain 12 UINT16 spectral bands representing SR scaled by 10000"
    return image.multiply(scale_factor)

In [ ]:
# Image filter parameters
cloudy_pixels_percentage = 10
tile_type = '10UDV'
years = range(2024, 2027)

# Prepare image sets
image_sets = []

for i in years:
    sentinel_set = get_image_collection(i, aoi, tile_type, cloudy_pixels_percentage)
    image_sets.append(sentinel_set)

In [ ]:
# Get image metadata from Google Earth Engine
def get_collection_metadata(image_collection):

    metadata =  {
        'No. of Images': image_collection.size().getInfo(),
        'system_index': image_collection.aggregate_array("system:index").getInfo(),
        'system_path': image_collection.aggregate_array('system:id').getInfo(),
        'band_names': image_collection.map(add_band_names).aggregate_array("band_names").getInfo(),
        'date': pd.to_datetime(image_collection.aggregate_array('system:time_start').getInfo(), unit="ms"),
        'crs': image_collection.map(get_crs).aggregate_array("crs").getInfo(),
        
        'PRODUCT_ID': image_collection.aggregate_array('PRODUCT_ID').getInfo(),
        'DATATAKE_IDENTIFIER': image_collection.aggregate_array('DATATAKE_IDENTIFIER').getInfo(),
        'MGRS_TILE': image_collection.aggregate_array('MGRS_TILE').getInfo(),
        'COVERAGE': f"{aoi.area(1).getInfo() / 1e6} km²",

        'CLOUDY_PIXEL_PERCENTAGE': image_collection.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo(),
        'SENSING_ORBIT_NUMBER': image_collection.aggregate_array('SENSING_ORBIT_NUMBER').getInfo(),
        "MEAN_SOLAR_ZENITH_ANGLE": image_collection.aggregate_array("MEAN_SOLAR_ZENITH_ANGLE").getInfo(),
        "MEAN_SOLAR_AZIMUTH_ANGLE": image_collection.aggregate_array("MEAN_SOLAR_AZIMUTH_ANGLE").getInfo(),
    }

    return metadata

In [ ]:
# View tabular image metadata
for image_set in image_sets:
    metadata_sets = get_collection_metadata(image_set)
    display(pd.DataFrame(metadata_sets))

In [ ]:
# Mask bad pixels, keeping 4: vegetation, 5: no vegetation
def SCL_mask(image):
    scl = image.select("SCL")
    good_pixels = scl.gte(4).And(scl.lte(5))
    return image.updateMask(good_pixels)

# Compute median over August images 
def median_composite(images, band_list):
    SCL_masked_images = images.map(SCL_mask)
    band_masked_images = SCL_masked_images.select(band_list)
    scale_masked_images = band_masked_images.map(scale_reflectance)

    source_projection = ee.Image(images.first()).select("B4").projection()

    return (
        scale_masked_images
        .median()
        .setDefaultProjection(source_projection)
    )

# Compute NDVI
def ndvi(images):
    ndvi = images.normalizedDifference(["B8", "B4"])

    return ndvi

In [ ]:
# Prepare datasets with median and NDVI
bands = ["B4", "B3", "B2", "B8"]
full_composite_sets = []

for image_set in image_sets:
    composite_set =  median_composite(image_set, bands)
    ndvi_set = ndvi(composite_set)
    full_composite_set = composite_set.addBands(ndvi_set)

    full_composite_sets.append(full_composite_set)

In [ ]:
# Preview image of interest (IOI)
try:
    img = full_composite_sets[0] # IOI
    url = img.select(["nd"]).getThumbURL({
        "region": aoi,                   
        "dimensions": 500,           
        "min": -1, 
        "max": 1,           
        "format": "png",
    })

    print(f"Generated URL: {url}\n")
    
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Server Error ({response.status_code}):")
        print(response.text) 
    else:
        print("URL is valid! Attempting display...")
        display(Image(url=url))

except Exception as e:
    print(f"Script execution failed: {e}")

In [ ]:
# Export image as task to Google Drive
def task(export_bands, year, aoi_bounds, tile):
    export_task = ee.batch.Export.image.toDrive(
        image=export_bands,
        description=f"sentinel_l2a_{year}08_{tile}_composite",
        folder="EarthEngine",
        fileNamePrefix=f"sentinel_l2a_{year}08_{tile}_composite",
        region=aoi_bounds,
        scale=10,
        crs="EPSG:32610",
        maxPixels=1e9,
        fileFormat="GeoTIFF", 
        formatOptions={"noData": -9999},
    )

    export_task.start()
    print("Export started")
    return export_task

# Select export bands
def sentinel_export(sentinel_bands):
    return sentinel_bands.select(
        ["B4", "B3", "B2", "B8", "nd"]
    ).toFloat()

In [ ]:
# Prepare export datasets and start exports
export_bands = []

for full_composite_set in full_composite_sets:
    export_band = sentinel_export(full_composite_set)
    export_bands.append(export_band)
    
tasks = [
    task(export_band, year, aoi, tile_type)
    for year, export_band in zip(years, export_bands)
]

In [ ]:
# Check export status
for year, export_task in zip(years, tasks):
    print(year, export_task.status())

### Get Dynamic World V1

In [25]:
# Get image from Google Earth Engine
def add_coverage(image):
    # Fraction of the AOI where this image has data (1.0 = full coverage)
    cover = (
        image.select("label").mask().unmask(0)
        .reduceRegion(ee.Reducer.mean(), aoi, 100, maxPixels=1e9)
        .get("label")
    )
    return image.set("aoi_coverage", cover)

dw = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterDate("2026-08-1", "2026-08-31")
    .filterBounds(aoi)
    .map(add_coverage)
    .filter(ee.Filter.gte("aoi_coverage", 0.95))
)

print( 
    f"Number of images: {dw.size().getInfo()}\n"
    f"Images: {dw.aggregate_array("system:index").getInfo()}\n"
)

Number of images: 2
Images: ['20260807T190911_20260807T191738_T10UDV', '20260807T190911_20260807T191738_T10UEV']



In [26]:
# Image filters
dw_image = dw.filter(
    ee.Filter.eq(
        "system:index",
        "20260807T190911_20260807T191738_T10UDV"
    )
).first()

# Get label metadata
print( 
    f"image ID: {dw_image.getInfo()["id"]}\n"
    f"bands: {dw_image.bandNames().getInfo()}\n"
)

image ID: GOOGLE/DYNAMICWORLD/V1/20260807T190911_20260807T191738_T10UDV
bands: ['water', 'trees', 'grass', 'flooded_vegetation', 'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice', 'label']



In [34]:
# Export image as task to Google Drive
dw_label = dw_image.select("label").unmask(255).toUint8()

task_dw = ee.batch.Export.image.toDrive(
    image=dw_label,
    description="dynamic_world_20260807_T10UDV",
    folder="EarthEngine",
    fileNamePrefix="dynamic_world_20260807_T10UDV",
    region=aoi,
    scale=10,
    crs="EPSG:32610",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={"noData": 255},
)

task_dw.start()
print("Export started")

Export started


In [52]:
print(task_dw.status())

{'state': 'COMPLETED', 'description': 'dynamic_world_20260807_T10UDV', 'priority': 100, 'creation_timestamp_ms': 1793914782362, 'update_timestamp_ms': 1793914961025, 'start_timestamp_ms': 1793914788044, 'task_type': 'EXPORT_IMAGE', 'destination_uris': ['https://drive.google.com/#folders/16PMye91hB5cePcHjvv1Tr8_qE50-lc8r'], 'attempt': 1, 'batch_eecu_usage_seconds': 18.00111198425293, 'id': 'XBXOG6J3YTOXGXBTPXVXGXZR', 'name': 'projects/gis-project-510419/operations/XBXOG6J3YTOXGXBTPXVXGXZR'}
